In [79]:
import sqlite3
import pandas as pd

In [80]:
pip install ipython-sql

Note: you may need to restart the kernel to use updated packages.


In [81]:
df = pd.read_csv('cleaned_data.csv')
conn = sqlite3.connect('superstore.db')
df.to_sql('superstore_sales', conn, if_exists='replace', index=False)


DatabaseError: Execution failed on sql 'DROP TABLE "superstore_sales"': database is locked

In [ ]:
%load_ext sql
%sql sqlite:///superstore.db
%config SqlMagic.displaylimit = 20

Connecting to 'sqlite:///superstore.db'

In [ ]:
%%sql   # see the table look like
SELECT *
FROM superstore_sales
LIMIT 10

Running query in 'sqlite:///superstore.db'

row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0.0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0.0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0.0,14.1694
7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0.0,1.9656
8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152
9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825
10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0.0,34.47


In [ ]:
%%sql       # how many total order numbers?
SELECT COUNT(DISTINCT order_id) as total_orders
FROM superstore_sales

Running query in 'sqlite:///superstore.db'

total_orders
5009


In [ ]:
%%sql       # how many region and state was the product sale?
SELECT COUNT(DISTINCT region) as number_of_region, COUNT(DISTINCT state) as number_of_state
FROM superstore_sales


Running query in 'sqlite:///superstore.db'

number_of_region,number_of_state
4,49


In [ ]:
%%sql   # how many product items are in each order?
SELECT order_id, COUNT(*) as total_product_items
FROM superstore_sales
GROUP BY order_id
ORDER BY order_id desc

Running query in 'sqlite:///superstore.db'

order_id,total_product_items
US-2017-169551,6
US-2017-169502,2
US-2017-169488,2
US-2017-169320,2
US-2017-168802,1
US-2017-168690,1
US-2017-168613,1
US-2017-168116,2
US-2017-167920,7
US-2017-167570,1


In [ ]:
%%sql   # which products are most propular/move the most unit?
SELECT category, product_name, SUM(quantity) as total_unit_sale
FROM superstore_sales
GROUP BY category, product_name
ORDER BY total_unit_sale DESC
LIMIT 5


Running query in 'sqlite:///superstore.db'

category,product_name,total_unit_sale
Office Supplies,Staples,215
Office Supplies,Staple envelope,170
Office Supplies,Easy-staple paper,150
Office Supplies,Staples in misc. colors,86
Furniture,KI Adjustable-Height Table,74


In [ ]:
%%sql       # which products bring the most revenue?
SELECT category, product_name, ROUND(SUM(sales), 2) as total_sales
FROM superstore_sales
GROUP BY category, product_name
ORDER BY total_sales DESC
LIMIT 5 


Running query in 'sqlite:///superstore.db'

category,product_name,total_sales
Technology,Canon imageCLASS 2200 Advanced Copier,61599.82
Office Supplies,Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind,27453.38
Technology,Cisco TelePresence System EX90 Videoconferencing Unit,22638.48
Furniture,HON 5400 Series Task Chairs for Big and Tall,21870.58
Office Supplies,GBC DocuBind TL300 Electric Binding System,19823.48


In [ ]:
%%sql   # which products bring most profitability?
SELECT product_name, 
    ROUND(SUM(profit), 2) as total_profit,
    ROUND(SUM(profit)*100.0/SUM(sales), 2) as profit_margin
FROM superstore_sales
GROUP BY product_name
ORDER BY total_profit DESC
LIMIT 5

Running query in 'sqlite:///superstore.db'

product_name,total_profit,profit_margin
Canon imageCLASS 2200 Advanced Copier,25199.93,40.91
Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind,7753.04,28.24
Hewlett Packard LaserJet 3310 Copier,6983.88,37.07
Canon PC1060 Personal Laser Copier,4570.93,39.34
"HP Designjet T520 Inkjet Large Format Printer - 24"" Color",4094.98,22.29


In [ ]:
%%sql   # which specific product or sub-categories have the lowest profit margin 
        # or lose money?
SELECT category, sub_category, product_name,
    ROUND(SUM(profit)*100.0/SUM(sales), 2) as profit_margin
FROM superstore_sales
GROUP BY category, sub_category, product_name
ORDER BY profit_margin 
LIMIT 5

Running query in 'sqlite:///superstore.db'

category,sub_category,product_name,profit_margin
Office Supplies,Appliances,Eureka Disposable Bags for Sanitaire Vibra Groomer I Upright Vac,-275.0
Furniture,Bookcases,"Bush Westfield Collection Bookcases, Dark Cherry Finish, Fully Assembled",-210.0
Office Supplies,Appliances,Euro Pro Shark Stick Mini Vacuum,-190.71
Technology,Machines,Okidata B401 Printer,-140.0
Technology,Machines,Zebra GK420t Direct Thermal/Thermal Transfer Printer,-133.33


In [ ]:
%%sql       # do the high sale products also result in high profit,
            # or do discounts eat up the profit?
SELECT category, product_name,
    ROUND(AVG(discount), 2) as avg_discount,
    ROUND(SUM(sales), 2) as total_sales,
    ROUND(SUM(profit), 2) as total_profit,
    ROUND(SUM(profit)*100.0/SUM(sales), 2) as profit_margin
FROM superstore_sales
GROUP BY category, product_name
ORDER BY total_sales DESC
LIMIT 15

Running query in 'sqlite:///superstore.db'

category,product_name,avg_discount,total_sales,total_profit,profit_margin
Technology,Canon imageCLASS 2200 Advanced Copier,0.12,61599.82,25199.93,40.91
Office Supplies,Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind,0.24,27453.38,7753.04,28.24
Technology,Cisco TelePresence System EX90 Videoconferencing Unit,0.5,22638.48,-1811.08,-8.0
Furniture,HON 5400 Series Task Chairs for Big and Tall,0.2,21870.58,0.0,0.0
Office Supplies,GBC DocuBind TL300 Electric Binding System,0.3,19823.48,2233.51,11.27
Office Supplies,GBC Ibimaster 500 Manual ProClick Binding System,0.52,19024.5,760.98,4.0
Technology,Hewlett Packard LaserJet 3310 Copier,0.2,18839.69,6983.88,37.07
Technology,"HP Designjet T520 Inkjet Large Format Printer - 24"" Color",0.17,18374.9,4094.98,22.29
Office Supplies,GBC DocuBind P400 Electric Binding System,0.45,17965.07,-1878.17,-10.45
Office Supplies,High Speed Automatic Electric Letter Opener,0.07,17030.31,-262.0,-1.54


In [ ]:
# High sales does not predict profitability. Among the top 15 products by total sales, profitability
# are splits almost evenly, yet, the dividing line is the discount level not the sale sizes.
# Every product averaging 40%+ discount lost money while products averaging below 25% were reliably profitable
# This confirms discounting, not the transaction volume is the primary cause of decline in profits across the business

In [ ]:
%%sql       # how do discounts impact the final profit per order?
SELECT order_id, 
    ROUND(AVG(discount), 2) as avg_discount,
    ROUND(SUM(sales), 2) as total_sales,
    ROUND(SUM(profit), 2) as total_profit,
    ROUND(SUM(profit)*100.0/SUM(sales), 2) as profit_margin
FROM superstore_sales
GROUP BY order_id
ORDER BY avg_discount DESC
LIMIT 15


Running query in 'sqlite:///superstore.db'

order_id,avg_discount,total_sales,total_profit,profit_margin
US-2017-155299,0.8,1.62,-4.47,-275.0
US-2017-152366,0.8,97.26,-243.16,-250.0
US-2017-148551,0.8,760.98,-1141.47,-150.0
US-2017-145863,0.8,5.63,-9.7,-172.39
US-2017-144582,0.8,43.37,-69.4,-160.0
US-2017-143028,0.8,11.36,-17.05,-150.0
US-2017-132206,0.8,5.94,-8.9,-150.0
US-2017-130603,0.8,11.65,-17.47,-150.0
US-2017-127341,0.8,12.13,-20.62,-170.0
US-2017-124926,0.8,9.32,-24.71,-265.0


In [ ]:
# The significant losses reaching up to -275% of the margin value were concentrated on the order 
# with an 80% discount. However, it is just a small transactions so the absolute impact 
# in monetary terms are limited with only rare exceptions

In [ ]:
%%sql       # how do shipping mode impact the final profit per order?
SELECT ship_mode, 
    ROUND(SUM(sales), 2) as total_sales,
    ROUND(SUM(profit), 2) as total_profit,
    ROUND(SUM(profit)*100.0/SUM(sales), 2) as profit_margin
FROM superstore_sales
GROUP BY ship_mode
ORDER BY profit_margin DESC


Running query in 'sqlite:///superstore.db'

ship_mode,total_sales,total_profit,profit_margin
First Class,351428.42,48969.84,13.93
Second Class,459193.57,57446.64,12.51
Same Day,128363.13,15891.76,12.38
Standard Class,1358215.74,164088.79,12.08


In [ ]:
# First class method generated the highest margin (13.93%) while Standard Class generated the highest 
# sales  but it has the lowest margin (12.08%), despite the gap margin values between First and Standard class are not
# particularly large

In [ ]:
%%sql   # how many product items, total sales and profits are in each order?
SELECT order_id, 
    COUNT(*) as total_orders, 
    ROUND(SUM(sales), 2) as total_sales,
    ROUND(SUM(profit), 2) as total_profit,
    ROUND(SUM(profit)*100.0/SUM(sales), 2) as profit_margin
FROM superstore_sales
GROUP BY order_id
ORDER BY order_id desc


Running query in 'sqlite:///superstore.db'

order_id,total_orders,total_sales,total_profit,profit_margin
US-2017-169551,6,1344.84,-62.29,-4.63
US-2017-169502,2,113.41,32.45,28.62
US-2017-169488,2,56.86,26.56,46.7
US-2017-169320,2,171.43,16.67,9.73
US-2017-168802,1,18.37,5.97,32.5
US-2017-168690,1,2.81,-1.97,-70.0
US-2017-168613,1,145.76,3.24,2.22
US-2017-168116,2,8167.42,-3825.34,-46.84
US-2017-167920,7,1827.51,764.88,41.85
US-2017-167570,1,215.54,-58.5,-27.14


In [ ]:
# as we can see the result above:
# US-2017-168116	2	8167.419999999999	-3825.3394000000003 it has the large sales but loss $3,825. 
# This order loss almost half its total sales (3825/8167=0.46 46%), 
# that make the analyst wonder that what products were in there, what discount level
# US-2017-169551	6	1344.838	-62.289500000000004 1344 in sales but still loss $62
# US-2017-168690	1	2.808	-1.9656
# US-2017-167920	7	1827.51	764.8837 profit 42%
# why do some product generate more sales but less proft 
# (big order aren't more profitable, small order aren't more profitable (US-2017-168690	1	2.808	-1.9656))

In [ ]:
%%sql   # what products were in US-2017-168116, what discount level that caused the large losses
SELECT order_id, category, sub_category, product_name, sales, profit, discount
FROM superstore_sales
WHERE order_id = 'US-2017-168116'

Running query in 'sqlite:///superstore.db'

order_id,category,sub_category,product_name,sales,profit,discount
US-2017-168116,Technology,Machines,Cubify CubeX 3D Printer Triple Head Print,7999.98,-3839.9904,0.5
US-2017-168116,Office Supplies,Appliances,"Eureka The Boss Plus 12-Amp Hard Box Upright Vacuum, Red",167.44,14.651,0.2


In [ ]:
# deep discount 50%, the reason make the sales less profit is because high discount = negative profit, not because high sales = low profit
# it discount half of the sale price, the product's cost basis is roughly equal to its full price
# usually whenever the discount is 50% or over then it will pushed it straight into loss territory

In [ ]:
%%sql 
SELECT ROUND(SUM(profit), 2) as total_profit,
    ROUND(SUM(profit)*100.0/SUM(sales), 2) as profit_margin
    
FROM superstore_sales
WHERE category = 'Furniture'

Running query in 'sqlite:///superstore.db'

total_profit,profit_margin
18451.27,2.49


In [ ]:
%%sql    # show the specific product that has the big profit
SELECT order_id, category, sub_category, product_name, sales, profit, discount
FROM superstore_sales
WHERE order_id = 'US-2017-167920'

Running query in 'sqlite:///superstore.db'

order_id,category,sub_category,product_name,sales,profit,discount
US-2017-167920,Office Supplies,Binders,"XtraLife ClearVue Slant-D Ring Binder, White, 3""",29.36,13.5056,0.0
US-2017-167920,Office Supplies,Appliances,Belkin F9M820V08 8 Outlet Surge,214.9,62.321,0.0
US-2017-167920,Office Supplies,Binders,"Avery Durable Slant Ring Binders, No Labels",15.92,7.4824,0.0
US-2017-167920,Technology,Accessories,Logitech ClearChat Comfort/USB Headset H390,146.45,48.3285,0.0
US-2017-167920,Office Supplies,Storage,Eldon Gobal File Keepers,15.14,0.6056,0.0
US-2017-167920,Office Supplies,Labels,Avery 492,5.76,2.6496,0.0
US-2017-167920,Technology,Copiers,Canon Imageclass D680 Copier / Fax,1399.98,629.991,0.0


In [ ]:
# based on the result, every single item has 0% discount and the total profit margin is roughly 42%. By contrast, the order id US-2017-168116 has one of the item at 50%,
# that's make sense that it loss, at 0% discount we can earn the profit equal to half of the sales(revenue)
# in general, more discount = more loss, less discount = less loss
# discount is the biggest factor that can turn a profit of the product into a loss
# when the cost/margin make the product is less, then you defintely get the huge profit even if you give a discount
# by contrast when the cost to make the product is high then you will get the tiny profit but if you give a discount you could easily start losing money
# however Binders (sales 29.36, profit 13.50 then earned 45%), Appliances (sales 214.9 profit 62.32 earned 28%), Copiers (sales 13999 profit 629 earn $.45%)
# even at 0% discount, the discount is not only the factor but if the revenue and the sales are not much then we can earned the profit


In [ ]:
%%sql   # what discount makes losses,  also group discount into the discount rate?
SELECT discount, 
    ROUND(SUM(profit) / SUM(sales) * 100, 2) as profit_margin,
    CASE 
        WHEN discount = 0 THEN 'No discount'
        WHEN discount <= 0.2 THEN 'Low discount'
        WHEN discount BETWEEN 0.2 AND 0.4 THEN 'Median discount'
        ELSE 'High discount'
    END AS discount_rate,
    ROUND(SUM(sales), 2) as total_sales, 
    ROUND(SUM(profit), 2) as total_profits
FROM superstore_sales
GROUP BY discount
ORDER BY total_profits

Running query in 'sqlite:///superstore.db'

discount,profit_margin,discount_rate,total_sales,total_profits
0.7,-98.66,High discount,40620.28,-40075.36
0.8,-180.03,High discount,16963.76,-30539.04
0.4,-19.81,Median discount,116417.78,-23057.05
0.5,-34.8,High discount,58918.54,-20506.43
0.3,-10.05,Median discount,103226.65,-10369.28
0.6,-89.46,High discount,6644.7,-5944.66
0.45,-45.45,High discount,5484.97,-2493.11
0.32,-16.5,Median discount,14493.46,-2391.14
0.15,5.15,Low discount,27558.52,1418.99
0.1,16.61,Low discount,54369.35,9029.18


In [ ]:
# (discount 70% has the highest loss money while 10% has the highest profit)
# discount above 15% turn the sales into losses, higher discounts are strongly 
# turn the sales into the negative profitability  

In [ ]:
%%sql   # which product categories are driving the losses
SELECT category,
    ROUND(SUM(sales), 2) as total_sales, 
    ROUND(SUM(profit), 2) as total_profits,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) as profit_margin
FROM superstore_sales
GROUP BY category
ORDER BY total_profits

Running query in 'sqlite:///superstore.db'

category,total_sales,total_profits,profit_margin
Furniture,741999.8,18451.27,2.49
Office Supplies,719047.03,122490.8,17.04
Technology,836154.03,145454.95,17.4


In [ ]:
# Technology generated the highest total sales ($836K) and highest total profit (145K) while the Furniture 
# total sales (742K) and had the lowest total profit (18K)
# however the total sales of the furniture is roughly close to the total sales of the office supplies
# but the office supplies generated the total profit is 122K while the funiture is around 18K, 
# that's a huge difference
# So Why does the Funiture generated almost 742K in sales but only 18K for profit?

In [ ]:
%%sql   # which sub_categories in Furniture are driving the losses
SELECT sub_category,     
    ROUND(SUM(sales), 2) as total_sales, 
    ROUND(SUM(profit), 2) as total_profits,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) as profit_margin
FROM superstore_sales
WHERE category = 'Furniture'
GROUP BY sub_category
ORDER BY total_profits

Running query in 'sqlite:///superstore.db'

sub_category,total_sales,total_profits,profit_margin
Tables,206965.53,-17725.48,-8.56
Bookcases,114880.0,-3472.56,-3.02
Furnishings,91705.16,13059.14,14.24
Chairs,328449.1,26590.17,8.1


In [ ]:
# as we can see the result above, the losses of Furniture driven from Tables and Bookcases. Although the 
# Furniture generated more sales than the Office Supplies, its profit was much lower because Tables 
# and Bookcases generated large negative profit

In [ ]:
%%sql   # why Tables lose money? see discount?
SELECT discount,
    COUNT(*) as number_of_items,
    ROUND(SUM(sales), 2) as total_sales, 
    ROUND(SUM(profit), 2) as total_profits,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) as profit_margin_as_percentage
FROM superstore_sales
WHERE sub_category = 'Tables'
GROUP BY discount
ORDER BY total_profits

Running query in 'sqlite:///superstore.db'

discount,number_of_items,total_sales,total_profits,profit_margin_as_percentage
0.4,75,45614.41,-16187.4,-35.49
0.5,36,13675.01,-8615.39,-63.0
0.3,54,25182.15,-3402.33,-13.51
0.45,11,5484.97,-2493.11,-45.45
0.2,71,45430.23,-303.56,-0.67
0.0,72,71578.76,13276.3,18.55


In [ ]:
# Table were profitable at 0% discount but profitability drops sharply as discounts increased; 
# at discount 40-50% Tables genereated significant losses

In [ ]:
%%sql     # which specific products are causing the biggest losses within Tables
SELECT product_name,
    COUNT(*) as number_of_items,
    ROUND(SUM(sales), 2) as total_sales, 
    ROUND(SUM(profit), 2) as total_profits,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) as profit_margin_as_percentage
FROM superstore_sales
WHERE sub_category = 'Tables'
GROUP BY product_name
ORDER BY total_profits

Running query in 'sqlite:///superstore.db'

product_name,number_of_items,total_sales,total_profits,profit_margin_as_percentage
Chromcraft Bull-Nose Wood Oval Conference Tables & Bases,5,9917.64,-2876.12,-29.0
Bush Advantage Collection Racetrack Conference Table,7,9544.73,-1934.4,-20.27
Balt Solid Wood Round Tables,4,6518.75,-1201.06,-18.42
BoxOffice By Design Rectangular and Half-Moon Meeting Room Tables,3,1706.25,-1148.44,-67.31
"Riverside Furniture Oval Coffee Table, Oval End Table, End Table with Drawer",5,4446.18,-1147.4,-25.81
Bretford Just In Time Height-Adjustable Multi-Task Work Tables,4,5634.9,-964.19,-17.11
"Bevis Oval Conference Table, Walnut",8,6942.07,-856.01,-12.33
BPI Conference Tables,5,2241.87,-795.97,-35.5
Hon 94000 Series Round Tables,5,7404.5,-681.21,-9.2
"Chromcraft Bull-Nose Wood 48"" x 96"" Rectangular Conference Tables",6,4297.64,-611.59,-14.23


In [ ]:
# Chromcraft has the largest total sale but biggest losses (-28 profit margin) 

In [ ]:
%%sql   # which caused the Chromcarft has biggest losses? discount?
SELECT discount,
    COUNT(*) as number_of_items,
    ROUND(SUM(sales), 2) as total_sales, 
    ROUND(SUM(profit), 2) as total_profits,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) as profit_margin_as_percentage
FROM superstore_sales
WHERE sub_category = 'Tables' and product_name = 'Chromcraft Bull-Nose Wood Oval Conference Tables & Bases'
GROUP BY discount
ORDER BY total_profits

Running query in 'sqlite:///superstore.db'

discount,number_of_items,total_sales,total_profits,profit_margin_as_percentage
0.4,3,6942.35,-3008.35,-43.33
0.2,1,1322.35,-99.18,-7.5
0.0,1,1652.94,231.41,14.0


In [ ]:
%%sql     # how much the Table losses are actually caused by high-discount orders?
SELECT product_name, discount,
    COUNT(*) AS number_of_items,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profits
FROM superstore_sales
WHERE sub_category = 'Tables' AND profit < 0
GROUP BY product_name, discount
ORDER BY total_profits

Running query in 'sqlite:///superstore.db'

product_name,discount,number_of_items,total_sales,total_profits
Chromcraft Bull-Nose Wood Oval Conference Tables & Bases,0.4,3,6942.35,-3008.35
Balt Solid Wood Round Tables,0.4,2,2946.83,-1522.53
"Riverside Furniture Oval Coffee Table, Oval End Table, End Table with Drawer",0.4,3,3958.53,-1187.56
Bush Advantage Collection Racetrack Conference Table,0.4,3,3054.31,-1119.91
Bush Advantage Collection Racetrack Conference Table,0.5,1,1272.63,-814.48
Hon 94000 Series Round Tables,0.4,1,1421.66,-734.53
BoxOffice By Design Rectangular and Half-Moon Meeting Room Tables,0.5,2,984.38,-728.44
"Bevis Oval Conference Table, Walnut",0.4,3,1879.06,-720.3
"Chromcraft 48"" x 96"" Racetrack Double Pedestal Table",0.5,2,1282.56,-718.23
"Chromcraft Bull-Nose Wood 48"" x 96"" Rectangular Conference Tables",0.4,3,1983.53,-694.23


In [ ]:
# within of sub_category Tables above, many of the largest losses occured at 40% 
# while 50% produce subtantial losses

In [ ]:
%%sql       # how much of Tables' total loss is caused by high discount orderes specificaly, group the discount into the group and see
SELECT 
    CASE
    WHEN discount < 0.3 THEN 'Low discount(<30%)'
    ELSE 'High discount(30%+)'
    END AS discount_group,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profits,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) AS profit_margin_as_percentage
FROM superstore_sales
WHERE sub_category = 'Tables'
GROUP BY discount_group


Running query in 'sqlite:///superstore.db'

discount_group,total_sales,total_profits,profit_margin_as_percentage
High discount(30%+),89956.54,-30698.22,-34.13
Low discount(<30%),117008.99,12972.74,11.09


In [ ]:
# Tables with discount 30% or more made up less than half of the Tables sales (43% of total sales of high and low discount (89956+117008=207), 89956/207=43%)
# but they loss 30698. Order with discount below 30% made about 12972 in profit. The loss from the high discount sales was so large that wiped out all the profit from the lower discount sales and caused the Tables 
# to lose money overall

In [ ]:
%%sql   # which region perform best margin?
SELECT region,
    ROUND(SUM(sales), 2) as total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) as profit_margin
FROM superstore_sales
GROUP BY region
ORDER BY profit_margin 

Running query in 'sqlite:///superstore.db'

region,total_sales,total_profit,profit_margin
Central,501239.89,39706.36,7.92
South,391721.91,46749.43,11.93
East,678781.24,91522.78,13.48
West,725457.82,108418.45,14.94


In [ ]:
%%sql   # which region, states perform best sales?
SELECT region, state,
    ROUND(SUM(sales), 2) as total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) as profit_margin
FROM superstore_sales
GROUP BY region, state 
ORDER BY total_sales DESC
LIMIT 5

Running query in 'sqlite:///superstore.db'

region,state,total_sales,total_profit,profit_margin
West,California,457687.63,76381.39,16.69
East,New York,310876.27,74038.55,23.82
Central,Texas,170188.05,-25729.36,-15.12
West,Washington,138641.27,33402.65,24.09
East,Pennsylvania,116511.91,-15559.96,-13.35


In [ ]:
%%sql   # which region, states perform best profit?
SELECT region, state,
    ROUND(SUM(sales), 2) as total_sales,
    ROUND(SUM(profit), 2) AS total_profits,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) as profit_margin
FROM superstore_sales
GROUP BY region, state 
ORDER BY total_profits DESC
LIMIT 5

Running query in 'sqlite:///superstore.db'

region,state,total_sales,total_profits,profit_margin
West,California,457687.63,76381.39,16.69
East,New York,310876.27,74038.55,23.82
West,Washington,138641.27,33402.65,24.09
Central,Michigan,76269.61,24463.19,32.07
South,Virginia,70636.72,18597.95,26.33


In [ ]:
%%sql   # which region, states perform best margin?
SELECT region, state,
    ROUND(SUM(sales), 2) as total_sales,
    ROUND(SUM(profit), 2) AS total_profits,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) as profit_margin
FROM superstore_sales
GROUP BY region, state 
ORDER BY profit_margin DESC
LIMIT 5

Running query in 'sqlite:///superstore.db'

region,state,total_sales,total_profits,profit_margin
East,District of Columbia,2865.02,1059.59,36.98
East,Delaware,27451.07,9977.37,36.35
Central,Minnesota,29863.15,10823.19,36.24
East,Maine,1270.53,454.49,35.77
Central,Indiana,53555.36,18382.94,34.33


In [ ]:
# based on all 3 queries above, what region, state perform best (high sale, high profit, high margin)?
# California is the standout performer, leading in both high total sales and total profit, 
# while it is not the top of margin but there is no state comes close on that scale
# several Central region states (Minnesota, Indiana, Michigan) post margin above 32%, showing that Central's weak
# regional average is driven by a few states severe underperformers (Texas) rather than a weakness across
# the entire region

In [ ]:
%%sql       # which locations generate strong sale 
SELECT region, category,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profits,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) as profit_margin
FROM superstore_sales
GROUP BY region, category
ORDER BY profit_margin

Running query in 'sqlite:///superstore.db'

region,category,total_sales,total_profits,profit_margin
Central,Furniture,163797.16,-2871.05,-1.75
East,Furniture,208291.2,3046.17,1.46
West,Furniture,252612.74,11504.95,4.55
Central,Office Supplies,167026.42,8879.98,5.32
South,Furniture,117298.68,6771.21,5.77
South,Technology,148771.91,19991.83,13.44
South,Office Supplies,125651.31,19986.39,15.91
West,Technology,251991.83,44303.65,17.58
East,Technology,264973.98,47462.04,17.91
Central,Technology,170416.31,33697.43,19.77


In [ ]:
# WEST is the strongest performance region which generating the high total sales and profit. In contrast, compare with Central and South, 
# Central generated more than South but earned the lowest profit of all four regions

In [ ]:
%%sql       # which category in the Central generated more sale but less profit
SELECT category,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profits,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) as profit_margin
FROM superstore_sales
WHERE region = 'Central'
GROUP BY category
ORDER BY profit_margin

Running query in 'sqlite:///superstore.db'

category,total_sales,total_profits,profit_margin
Furniture,163797.16,-2871.05,-1.75
Office Supplies,167026.42,8879.98,5.32
Technology,170416.31,33697.43,19.77


In [ ]:
# All 3 categories have the sales iin 163K-170K but Furniture is the only one 
# that losing money. Office Suplies and Technology both turn a profit, just Furniture flips negative

In [82]:
%%sql       # which Central states losing money
SELECT state,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) as profit_margin
FROM superstore_sales
WHERE region = 'Central'
GROUP BY state
ORDER BY total_profit

Running query in 'sqlite:///superstore.db'

state,total_sales,total_profit,profit_margin
Texas,170188.05,-25729.36,-15.12
Illinois,80166.1,-12607.89,-15.73
North Dakota,919.91,230.15,25.02
South Dakota,1315.56,394.83,30.01
Kansas,2914.31,836.44,28.7
Iowa,4579.76,1183.81,25.85
Nebraska,7464.93,2037.09,27.29
Oklahoma,19683.39,4853.96,24.66
Missouri,22205.15,6436.21,28.99
Wisconsin,32114.61,8401.8,26.16


In [ ]:
# Texas and Illinois were the only states in the Central region that generated
# an overall loss while other Central states remained profitable
# Texas had the largest total loss 257K while Illinois had 126K  

In [ ]:
%%sql       
SELECT state, category,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profits,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) as profit_margin
FROM superstore_sales
WHERE state IN ('Texas', 'Illinois')
GROUP BY state, category
ORDER BY total_profits


Running query in 'sqlite:///superstore.db'

state,category,total_sales,total_profits,profit_margin
Texas,Office Supplies,44490.53,-18584.64,-41.77
Texas,Furniture,60593.29,-10436.14,-17.22
Illinois,Furniture,28274.52,-9076.29,-32.1
Illinois,Office Supplies,19907.91,-8354.16,-41.96
Texas,Technology,65104.22,3291.43,5.06
Illinois,Technology,31983.67,4822.56,15.08


In [ ]:
# Texas	Office Supplies	44490.53	-18584.64	-41.77
# Illinois	Office Supplies	19907.91	-8354.16	-41.96 
# Office Supplies in Texas has loss 18K (41%) but the Office Supplies in Illinois has loss 
# less than the Texas 8K but still 41%.  Office Supplies was higly unprofitable in both Texas and Illinois
# but Texas has more sales Office Supplies but the profit margins are almost identical at about 42% in both states

In [ ]:
%%sql
SELECT state, discount,
       COUNT(*) as num_items,
       ROUND(SUM(sales), 2) as total_sales,
       ROUND(SUM(profit), 2) as total_profit,
       ROUND(SUM(profit) / SUM(sales) * 100, 2) as profit_margin
FROM superstore_sales
WHERE state IN ('Texas', 'Illinois') AND category = 'Office Supplies'
GROUP BY state, discount
ORDER BY state, discount

Running query in 'sqlite:///superstore.db'

state,discount,num_items,total_sales,total_profit,profit_margin
Illinois,0.2,185,14394.64,1332.59,9.26
Illinois,0.8,100,5513.27,-9686.74,-175.7
Texas,0.2,404,33040.04,2267.65,6.86
Texas,0.8,200,11450.49,-20852.3,-182.11


In [ ]:
# Texas and Illinois are the only two states in Central with negative profit. In both, Office Supplies orders discounted at 80%
# loss between 175% and 182% for every dollar of sales, while the same category discounted at 20%
# remained the positive profitable in both states. This confirm that extreme discounting is the direct cause of Central's regional loss,
# not the product category or state itself

In [ ]:
%%sql       # which customer segment bring in the most revenue?
SELECT segment, 
       ROUND(SUM(sales), 2) as total_sales,
       ROUND(SUM(profit), 2) as total_profit,
       ROUND(SUM(profit)*100.0/SUM(sales), 2) as profit_margin
FROM superstore_sales
GROUP BY segment
ORDER BY total_sales DESC

Running query in 'sqlite:///superstore.db'

segment,total_sales,total_profit,profit_margin
Consumer,1161401.34,134119.21,11.55
Corporate,706146.37,91979.13,13.03
Home Office,429653.15,60298.68,14.03


In [ ]:
# Consumer is the largest customer segment within three segments. It generated $1.16M in sales and $134K profit
# However it has the lowest profit margin at (11.55%)
# Home Office has the lowest sale and profit but it has the largest profit margin (14.03%),
# indicating that it generated more profit per dollar of sales


In [ ]:
%%sql       # who are the top customers by total purchase value?
SELECT customer_id, customer_name,
    COUNT(DISTINCT order_id) as num_orders,
    ROUND(SUM(sales), 2) as total_purchase_values,
    ROUND(SUM(profit), 2) as total_profit,
    ROUND(SUM(profit)*100.0/SUM(sales), 2) as profit_margin
FROM superstore_sales
GROUP BY customer_id, customer_name
ORDER BY total_purchase_values DESC
LIMIT 5

Running query in 'sqlite:///superstore.db'

customer_id,customer_name,num_orders,total_purchase_values,total_profit,profit_margin
SM-20320,Sean Miller,5,25043.05,-1980.74,-7.91
TC-20980,Tamara Chand,5,19052.22,8981.32,47.14
RB-19360,Raymond Buch,6,15117.34,6976.1,46.15
TA-21385,Tom Ashbrook,4,14595.62,4703.79,32.23
AB-10105,Adrian Barton,10,14473.57,5444.81,37.62


In [ ]:
# Sean Miller is the highest of purchasing product but the total order is lowest which is 5 orders but generated
# the largest purchased values but loss profit

In [ ]:
%%sql       # how have sales and profits changed over time? (is sales growing year over year?)
SELECT strftime('%Y', order_date) as year, 
       ROUND(SUM(sales), 2) as total_sales,
       ROUND(SUM(profit), 2) as total_profit,
       ROUND(SUM(profit)*100.0/SUM(sales), 2) as profit_margin
FROM superstore_sales
GROUP BY year


Running query in 'sqlite:///superstore.db'

year,total_sales,total_profit,profit_margin
2014,484247.5,49543.97,10.23
2015,470532.51,61618.6,13.1
2016,609205.6,81795.17,13.43
2017,733215.26,93439.27,12.74


In [ ]:
# the sales and profits generally increased, the most recent period recording the highest sales
# and profits was 2017. However it was not the highest profit margin, its profit margin declined 
# from 13.43% in 2016 to 12.74% in 2017. This suggesting that the additional sales 2017 were slightly less profitable

In [ ]:
%%sql       # How did monthly sales and profitability change over time, 
            # and were there months with strong sales but weak profitability

SELECT
    strftime('%Y-%m', order_date) AS year_month,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) AS profit_margin
FROM superstore_sales
GROUP BY year_month
ORDER BY year_month

Running query in 'sqlite:///superstore.db'

year_month,total_sales,total_profit,profit_margin
2014-01,14236.9,2450.19,17.21
2014-02,4519.89,862.31,19.08
2014-03,55691.01,498.73,0.9
2014-04,28295.35,3488.84,12.33
2014-05,23648.29,2738.71,11.58
2014-06,34595.13,4976.52,14.39
2014-07,33946.39,-841.48,-2.48
2014-08,27909.47,5318.1,19.05
2014-09,81777.35,8328.1,10.18
2014-10,31453.39,3448.26,10.96


In [ ]:
# Somw high-sales month do not necessarily generate high profit margin
# This suggests that sales growth alone is not enough to evaluate business
# performance; profitability should also be monitored.

In [ ]:
%%sql       # monthly trend over time (combine all each month 2014-2017), 4 Jan (2014-2017)
SELECT strftime('%m', order_date) AS month,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) AS profit_margin
FROM superstore_sales
GROUP BY month
ORDER BY month


Running query in 'sqlite:///superstore.db'

month,total_sales,total_profit,profit_margin
01,94924.84,9134.45,9.62
02,59751.25,10294.61,17.23
03,205005.49,28594.69,13.95
04,137762.13,11587.44,8.41
05,155028.81,22411.31,14.46
06,152718.68,21285.8,13.94
07,147238.1,13832.66,9.39
08,159044.06,21776.94,13.69
09,307649.95,36857.48,11.98
10,200322.98,31784.04,15.87


In [ ]:
# Sep generated substantially more sales than the other month shown and also the highest total profit
# but it is not a largest profit margin 
# Feb has relatively low sales but is very different at converting those sales into profit
# It has the largest profit magin 
# April generated more than twice Feb's sales but only slightly more profit

In [ ]:
# Nov generated the highest total sales at $352K. However, Dec generated the highest total profit 
# at $43K desipte lower sales. Feb had relatively low sales but is very different at converting those sales into profit.
# It had the highest profit margion at 17.23%, showing that more sales don't always mean better profitability

In [ ]:
%%sql
SELECT 
    strftime('%m', order_date) as month,
    ROUND(AVG(discount), 2) as avg_discount
FROM superstore_sales
GROUP BY month
ORDER BY month

Running query in 'sqlite:///superstore.db'

month,avg_discount
01,0.15
02,0.15
03,0.16
04,0.16
05,0.17
06,0.16
07,0.16
08,0.15
09,0.15
10,0.16
